In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import numpy as np
import pandas as pd

Mounted at /content/drive


In [2]:
BASE_DIR = "/content/drive/MyDrive/thesis_movie_project"
PROCESSED_DIR = f"{BASE_DIR}/processed"

MASTER_PATH = f"{PROCESSED_DIR}/thesis_master_final.csv"

master = pd.read_csv(MASTER_PATH, low_memory=False)

print("Loaded master shape:", master.shape)
master.head(3)

Loaded master shape: (89569, 22)


,id,movieId,imdb_id,title,overview,keywords_clean,cast_clean,director_name,genres_clean,original_language,...,budget,revenue,roi_raw,runtime,popularity,vote_average,vote_count,text_overview_only,text_semantic_enriched,text_semantic_full
0,2,4470.0,tt0094675,Ariel,Taisto Kasurinen is a Finnish coal miner whose...,underdog prison factory worker prisoner helsin...,Turo Pajala Susanna Haavisto Matti Pellonpää E...,Aki Kaurismäki,Drama Crime,fi,...,NaN,0.0,NaN,69.0,3.860491,7.1,44.0,Taisto Kasurinen is a Finnish coal miner whose...,Ariel underdog prison factory worker prisoner ...,Ariel underdog prison factory worker prisoner ...
1,3,61724.0,tt0092149,Shadows in Paradise,"An episode in the life of Nikander, a garbage ...",salesclerk helsinki garbage independent film,Matti Pellonpää Kati Outinen Sakari Kuosmanen ...,Aki Kaurismäki,Drama Comedy,fi,...,NaN,0.0,NaN,76.0,2.292110,7.1,35.0,"An episode in the life of Nikander, a garbage ...",Shadows in Paradise salesclerk helsinki garbag...,Shadows in Paradise salesclerk helsinki garbag...
2,5,18.0,tt0113101,Four Rooms,It's Ted the Bellhop's first night on the job....,hotel new year's eve witch bet hotel room sper...,Tim Roth Antonio Banderas Jennifer Beals Madon...,Allison Anders,Crime Comedy,en,...,4000000.0,4300000.0,1.075,98.0,9.026586,6.5,539.0,It's Ted the Bellhop's first night on the job....,Four Rooms hotel new year's eve witch bet hote...,Four Rooms hotel new year's eve witch bet hote...


In [3]:
def clean_text_col(series):
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

placeholder_values = {
    "Unknown", "unknown", "UNKNOWN",
    "nan", "NaN", "None", "NONE",
    "N/A", "n/a", "null", "NULL"
}

text_cols = [
    "title", "overview", "keywords_clean", "cast_clean",
    "director_name", "genres_clean",
    "text_overview_only", "text_semantic_enriched", "text_semantic_full"
]

for col in text_cols:
    if col in master.columns:
        master[col] = clean_text_col(master[col])
        master[col] = master[col].replace(list(placeholder_values), "")

# numeric safety
for col in ["release_year", "runtime", "vote_count", "popularity", "budget", "revenue"]:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors="coerce")

# release date safety
if "release_date" in master.columns:
    master["release_date"] = pd.to_datetime(master["release_date"], errors="coerce")

# helper columns
master["overview_len"] = master["overview"].fillna("").astype(str).str.len()
master["overview_word_count"] = master["overview"].fillna("").astype(str).str.split().str.len()

master["budget_clean"] = pd.to_numeric(master["budget"], errors="coerce")
master["revenue_clean"] = pd.to_numeric(master["revenue"], errors="coerce")

master.loc[master["budget_clean"] <= 0, "budget_clean"] = np.nan
master.loc[master["revenue_clean"] <= 0, "revenue_clean"] = np.nan

master["roi_clean"] = np.where(
    master["budget_clean"].notna() & master["revenue_clean"].notna(),
    master["revenue_clean"] / master["budget_clean"],
    np.nan
)

# rebuild text columns after placeholder cleaning
master["text_overview_only"] = clean_text_col(master["overview"])

master["text_semantic_enriched"] = (
    clean_text_col(master["title"]) + " " +
    clean_text_col(master["keywords_clean"]) + " " +
    clean_text_col(master["overview"])
).str.replace(r"\s+", " ", regex=True).str.strip()

master["text_semantic_full"] = (
    clean_text_col(master["title"]) + " " +
    clean_text_col(master["keywords_clean"]) + " " +
    clean_text_col(master["cast_clean"]) + " " +
    clean_text_col(master["overview"])
).str.replace(r"\s+", " ", regex=True).str.strip()

print("Basic cleaning complete.")

Basic cleaning complete.


In [4]:
# ===============================
# HARMONIZED THESIS RULES
# ===============================

YEAR_MIN = 1950          # change to 1900 if you truly want older cinema included
YEAR_MAX = 2023          # stable cutoff; avoids incomplete freshest years
MIN_RUNTIME = 60
MIN_OVERVIEW_WORDS = 25
MIN_VOTE_COUNT = 5
MIN_POPULARITY = 2.0
DROP_TV_MOVIE = True

print("Rules:")
print("YEAR_MIN =", YEAR_MIN)
print("YEAR_MAX =", YEAR_MAX)
print("MIN_RUNTIME =", MIN_RUNTIME)
print("MIN_OVERVIEW_WORDS =", MIN_OVERVIEW_WORDS)
print("MIN_VOTE_COUNT =", MIN_VOTE_COUNT)
print("MIN_POPULARITY =", MIN_POPULARITY)
print("DROP_TV_MOVIE =", DROP_TV_MOVIE)

Rules:
YEAR_MIN = 1950
YEAR_MAX = 2023
MIN_RUNTIME = 60
MIN_OVERVIEW_WORDS = 25
MIN_VOTE_COUNT = 5
MIN_POPULARITY = 2.0
DROP_TV_MOVIE = True


In [5]:
df_semantic_analysis = master.copy()

# 1. basic movie-like constraints
df_semantic_analysis = df_semantic_analysis[
    df_semantic_analysis["release_year"].between(YEAR_MIN, YEAR_MAX, inclusive="both")
].copy()

df_semantic_analysis = df_semantic_analysis[
    df_semantic_analysis["runtime"].fillna(0) >= MIN_RUNTIME
].copy()

df_semantic_analysis = df_semantic_analysis[
    df_semantic_analysis["title"].fillna("").astype(str).str.len() > 0
].copy()

df_semantic_analysis = df_semantic_analysis[
    df_semantic_analysis["overview_word_count"].fillna(0) >= MIN_OVERVIEW_WORDS
].copy()

# 2. optional cinema-only cleanup
if DROP_TV_MOVIE:
    df_semantic_analysis = df_semantic_analysis[
        ~df_semantic_analysis["genres_clean"].fillna("").str.contains(r"\bTV Movie\b", regex=True)
    ].copy()

# 3. minimum documentation / visibility rule
visibility_mask = (
    (df_semantic_analysis["vote_count"].fillna(0) >= MIN_VOTE_COUNT) |
    (df_semantic_analysis["popularity"].fillna(0) >= MIN_POPULARITY) |
    (df_semantic_analysis["revenue_clean"].notna())
)

df_semantic_analysis = df_semantic_analysis[visibility_mask].copy()

# 4. dedupe safety
df_semantic_analysis = (
    df_semantic_analysis
    .drop_duplicates(subset="id")
    .sort_values(["release_year", "title"])
    .reset_index(drop=True)
)

print("df_semantic_analysis shape:", df_semantic_analysis.shape)
df_semantic_analysis.head(3)

df_semantic_analysis shape: (42709, 27)


,id,movieId,imdb_id,title,overview,keywords_clean,cast_clean,director_name,genres_clean,original_language,...,vote_average,vote_count,text_overview_only,text_semantic_enriched,text_semantic_full,overview_len,overview_word_count,budget_clean,revenue_clean,roi_clean
0,37087,105406.0,tt0042176,711 Ocean Drive,The Horatio Alger parable gets the film noir t...,gambling criminal syndicate bookie horse track...,Edmond O'Brien Joanne Dru Otto Kruger Barry Ke...,Joseph M. Newman,Drama Crime,en,...,6.1,8.0,The Horatio Alger parable gets the film noir t...,711 Ocean Drive gambling criminal syndicate bo...,711 Ocean Drive gambling criminal syndicate bo...,539,85,NaN,NaN,NaN
1,33474,71560.0,tt0042179,Abbott and Costello in the Foreign Legion,Jonesy and Lou are in Algeria looking for a wr...,foreign legion wrestler,Bud Abbott Lou Costello Patricia Medina Marc L...,Charles Lamont,Comedy,en,...,6.7,12.0,Jonesy and Lou are in Algeria looking for a wr...,Abbott and Costello in the Foreign Legion fore...,Abbott and Costello in the Foreign Legion fore...,311,51,NaN,NaN,NaN
2,705,926.0,tt0042192,All About Eve,From the moment she glimpses her idol at the s...,jealousy obsession manipulation fan engagement...,Bette Davis Anne Baxter George Sanders Celeste...,Joseph L. Mankiewicz,Drama,en,...,8.0,367.0,From the moment she glimpses her idol at the s...,All About Eve jealousy obsession manipulation ...,All About Eve jealousy obsession manipulation ...,424,71,1400000.0,63463.0,0.045331


In [6]:
df_revenue_analysis = df_semantic_analysis[
    df_semantic_analysis["revenue_clean"].notna()
].copy()

df_revenue_analysis["log_revenue"] = np.log1p(df_revenue_analysis["revenue_clean"])

df_roi_analysis = df_semantic_analysis[
    df_semantic_analysis["budget_clean"].notna() &
    df_semantic_analysis["revenue_clean"].notna()
].copy()

df_roi_analysis["log_budget"] = np.log1p(df_roi_analysis["budget_clean"])
df_roi_analysis["log_revenue"] = np.log1p(df_roi_analysis["revenue_clean"])

print("df_revenue_analysis shape:", df_revenue_analysis.shape)
print("df_roi_analysis shape:", df_roi_analysis.shape)

df_revenue_analysis shape: (8603, 28)
df_roi_analysis shape: (5707, 29)


In [7]:
print("Missing values (%) in df_semantic_analysis:")
print((df_semantic_analysis.isna().mean() * 100).round(2).sort_values(ascending=False).head(15))

print("\nYear coverage:")
movies_per_year = (
    df_semantic_analysis["release_year"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)
movies_per_year.columns = ["release_year", "movie_count"]
print(movies_per_year.tail(20))

print("\nMedian values:")
print("Median overview words:", df_semantic_analysis["overview_word_count"].median())
print("Median vote_count:", df_semantic_analysis["vote_count"].median())
print("Median popularity:", df_semantic_analysis["popularity"].median())

Missing values (%) in df_semantic_analysis:
roi_clean            86.64
roi_raw              81.41
revenue_clean        79.86
budget               78.37
budget_clean         78.37
release_date         47.44
movieId              47.44
revenue              41.56
imdb_id               4.27
id                    0.00
title                 0.00
original_language     0.00
genres_clean          0.00
director_name         0.00
cast_clean            0.00
dtype: float64

Year coverage:
    release_year  movie_count
54          2004          553
55          2005          627
56          2006          721
57          2007          735
58          2008          814
59          2009          943
60          2010          897
61          2011          954
62          2012          961
63          2013         1069
64          2014         1128
65          2015         1003
66          2016          837
67          2017         3032
68          2018         3172
69          2019         3124
70        

In [9]:
SEMANTIC_PATH = f"{PROCESSED_DIR}/df_semantic_analysis.csv"
REVENUE_PATH = f"{PROCESSED_DIR}/df_revenue_analysis.csv"
ROI_PATH = f"{PROCESSED_DIR}/df_roi_analysis.csv"
YEAR_COUNT_PATH = f"{PROCESSED_DIR}/movies_per_year_semantic_analysis.csv"

df_semantic_analysis.to_csv(SEMANTIC_PATH, index=False)
df_revenue_analysis.to_csv(REVENUE_PATH, index=False)
df_roi_analysis.to_csv(ROI_PATH, index=False)
movies_per_year.to_csv(YEAR_COUNT_PATH, index=False)

print("Saved:")
print("-", SEMANTIC_PATH)
print("-", REVENUE_PATH)
print("-", ROI_PATH)
print("-", YEAR_COUNT_PATH)

Saved:
- /content/drive/MyDrive/thesis_movie_project/processed/df_semantic_analysis.csv
- /content/drive/MyDrive/thesis_movie_project/processed/df_revenue_analysis.csv
- /content/drive/MyDrive/thesis_movie_project/processed/df_roi_analysis.csv
- /content/drive/MyDrive/thesis_movie_project/processed/movies_per_year_semantic_analysis.csv


In [8]:
print("FINAL STATUS")
print("------------")
print("Master archive:", master.shape)
print("Semantic analysis dataset:", df_semantic_analysis.shape)
print("Revenue analysis dataset:", df_revenue_analysis.shape)
print("ROI analysis dataset:", df_roi_analysis.shape)

print("\nUse this file from now on for the main thesis network:")
print("df_semantic_analysis.csv")

FINAL STATUS
------------
Master archive: (89569, 27)
Semantic analysis dataset: (42709, 27)
Revenue analysis dataset: (8603, 28)
ROI analysis dataset: (5707, 29)

Use this file from now on for the main thesis network:
df_semantic_analysis.csv
